# 飲食店Tips：チップ額・チップ率の関連分析

run_id: `v020-exp-04`。Kaggle `jsphyg/tipping` をAPIで検索・取得し、外部CSVミラーは使用しない。

## 分析計画

1. 出所・取得日時・SHA-256を固定し、列定義と不明点をmanifestに記録する。
2. 欠損、型、範囲、カテゴリ、重複、チップ率の極端値を確認する。観測識別子がないため完全一致行は自動削除しない。
3. チップ額と会計額／人数の相関、チップ率（100 × tip / total_bill）と各特徴の関係を図表で示す。会計ごとの平均率と金額加重率を区別する。
4. 不均衡なカテゴリ・交絡を確認し、HC3頑健標準誤差付き回帰と仕様変更の感度分析を行う。探索的p値は確認的証拠とせず、カテゴリ検定には多重比較補正を使う。
5. 初回結果から実質的な未解決点を選び最大3回反復する。最後に新しいカーネルで全コードセルを上から再実行し、証拠参照、visual_audit、ライフサイクルを確認する。

## 因果解釈の範囲

観察データの記述的・関連的分析のみ。サービス品質、店・担当者、顧客収入、再来店、日付などの未観測交絡、非無作為抽出、率の分母共有を明示する。

In [1]:
from pathlib import Path
import os, sys, json, hashlib, io, contextlib, warnings, zipfile
from datetime import datetime, timezone
from dataclasses import asdict
import nbformat
ROOT = Path('/home/nahisaho/GitHub/jupytermind')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark/projects')
from ai_data_scientist import project_manager, lifecycle, language_router
RUN_ID = 'v020-exp-04'
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-04-tips')
project_manager.ensure_notebook(handle)
data_dir = project_manager.ensure_data_dir(handle)
lifecycle.register_run(RUN_ID, handle.notebook_path)
language = language_router.detect_language('チップ額とチップ率に関係する特徴を日本語で分析してください。')
phase_log = []
@contextlib.contextmanager
def phase(name, write=False):
    if lifecycle.is_cancel_requested(RUN_ID):
        raise RuntimeError('実行の取消しが要求されました')
    start = lifecycle.mark_write_start if write else lifecycle.mark_execution_start
    end = lifecycle.mark_write_end if write else lifecycle.mark_execution_end
    start(RUN_ID)
    phase_log.append({'phase': name, 'event': 'start', 'utc': datetime.now(timezone.utc).isoformat()})
    try:
        yield
    except Exception:
        lifecycle.mark_failed(RUN_ID)
        phase_log.append({'phase': name, 'event': 'failed'})
        raise
    finally:
        end(RUN_ID)
        phase_log.append({'phase': name, 'event': 'end', 'utc': datetime.now(timezone.utc).isoformat()})
print({'run_id': RUN_ID, 'language': language, 'state': lifecycle.get_run_status(RUN_ID).state})


{'run_id': 'v020-exp-04', 'language': 'ja', 'state': 'running'}


## Kaggle API検索・取得ログ

認証情報は表示・保存しない。例外本文も秘密情報が含まれる可能性があるため保存せず、失敗分類だけ記録する。再実行時もAPIから再取得し、SHA-256を確認する。

検索の最初の20件だけでは指定slugが見つからなかったため、owner限定検索と指定slugへの直接ファイルAPIで確認する。これは検索網羅性の修正であり、別データ／外部ソースへの切替ではない。

In [2]:
provenance = None
with phase('Kaggle公開データ検索・取得'):
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api_status = {'search': 'tipping', 'requested_ref': 'jsphyg/tipping',
                  'attempted_at_utc': datetime.now(timezone.utc).isoformat(), 'operation': 'authenticate'}
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            api.authenticate()
            api_status['authenticated'] = True
            api_status['operation'] = 'dataset_list(search=tipping)'
            matches = api.dataset_list(search='tipping')
        public_matches = [{'ref': str(d.ref), 'title': str(d.title)} for d in matches]
        api_status['matches'] = public_matches
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            api_status['operation'] = 'dataset_list(search=tips,user=jsphyg)'
            owner_matches = api.dataset_list(search='tips', user='jsphyg')
        api_status['owner_search'] = [{'ref': str(d.ref), 'title': str(d.title)} for d in owner_matches]
        # 検索先頭ページへの未掲載は不存在を意味しない。指定slugのファイルAPIを直接確認する。
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            api_status['operation'] = 'dataset_list_files(jsphyg/tipping)'
            listing = api.dataset_list_files('jsphyg/tipping')
        file_names = [str(f.name) for f in listing.files]
        api_status['files'] = file_names
        candidates = [name for name in file_names if name.lower().endswith('.csv') and 'tip' in name.lower()]
        if len(candidates) != 1:
            raise LookupError('tips_csv_not_uniquely_identified')
        file_name = candidates[0]
        if Path(file_name).name != file_name:
            raise ValueError('unexpected_nested_file_name')
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            api.dataset_download_file('jsphyg/tipping', file_name, path=str(data_dir), force=True, quiet=True)
        csv_path = data_dir / file_name
        zip_path = data_dir / (file_name + '.zip')
        if not csv_path.exists() and zip_path.exists():
            with zipfile.ZipFile(zip_path) as archive:
                matching_members = [name for name in archive.namelist() if Path(name).name == file_name]
                if len(matching_members) != 1:
                    raise ValueError('download_archive_member_not_unique')
                csv_path.write_bytes(archive.read(matching_members[0]))
        if not csv_path.is_file():
            raise FileNotFoundError('downloaded_csv_not_found')
        provenance = {'owner': 'jsphyg', 'dataset_slug': 'tipping', 'ref': 'jsphyg/tipping',
                      'kaggle_url': 'https://www.kaggle.com/datasets/jsphyg/tipping',
                      'file_name': file_name, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
                      'sha256': hashlib.sha256(csv_path.read_bytes()).hexdigest(), 'bytes': csv_path.stat().st_size,
                      'retrieval_method': 'Kaggle Python SDK; no external CSV mirror'}
    except Exception as exc:
        lifecycle.mark_failed(RUN_ID)
        api_status['status'] = 'failed'
        api_status['exception_class'] = type(exc).__name__
        api_status['http_status'] = getattr(getattr(exc, 'response', None), 'status_code', None)
        api_status['stop_reason'] = '指定データのAPI取得不可。対応する指定公開データを確認できないため実験停止。別ownerの候補や外部ソースへの置換は行わない。'
        display(api_status)
    else:
        api_status['status'] = 'success'
        display(api_status)
        display(provenance)


{'search': 'tipping',
 'requested_ref': 'jsphyg/tipping',
 'attempted_at_utc': '2026-10-01T16:10:16.802184+00:00',
 'operation': 'dataset_list_files(jsphyg/tipping)',
 'authenticated': True,
 'matches': [{'ref': 'ranaghulamnabi/restaurant-tips-dataset',
   'title': 'Restaurant Tips Dataset'},
  {'ref': 'thomasmann111111/fly-tipping', 'title': 'Fly Tipping'},
  {'ref': 'stoney71/aflstats',
   'title': 'Australian Football League (AFL) Database'},
  {'ref': 'coderanand/tips-dataset', 'title': 'Tips_Dataset'},
  {'ref': 'saurabhbadole/restaurant-tips-dataset',
   'title': 'Restaurant Tips Dataset'},
  {'ref': 'nudratabbas/global-supply-chain-risk-and-logistics-2024-2026',
   'title': 'Global Supply Chain Risk & Logistics (2024-2026)'},
  {'ref': 'yukikatherina/indonesian-social-media-toxicity-dataset',
   'title': 'Indonesian Social Media Toxicity Dataset'},
  {'ref': 'vishakhdapat/waiter-tip-prediction',
   'title': 'Waiter Tip Prediction'},
  {'ref': 'likithagedipudi/war-economic-and-li

**実行結果：データ取得不可による停止**

認証と一般検索は成功したが、指定 `jsphyg/tipping` のファイル一覧APIはHTTP 403 を返した。owner限定検索は空だった。公開取得可能な指定データを確認できず、外部CSVや別ownerの類似データを用いず実験を停止した。HTTP 403はアクセス拒否を示すが、削除・非公開化・権限制約のいずれかをこのログだけでは特定できない。

チップ額・チップ率の関係、効果量、有意性、図表は未生成。通常知られているTipsの統計や別データの数字で補完しない。

```evidence
{"execution_count": 2, "cited_value": "403", "claim_type": "descriptive"}
```

## データ定義manifest・分析前提manifest・適用可否

取得できていないファイルの列定義、標本数、通貨単位を検証済みと扱わない。取得日・ファイル名・SHA-256は未取得につきnull。試行日時は取得日時ではない。`target_available` 前提の不成立を警告として報告する。

In [3]:
from ai_data_scientist import data_definition, analysis_assumptions
with phase('未取得データの定義・分析前提manifest'):
    if api_status['status'] != 'failed':
        raise RuntimeError('取得条件が変わったため、未取得manifestではなく分析を再計画してください')
    FV = data_definition.FieldValue
    data_manifest = data_definition.build_manifest(
        source={
            'owner': FV('jsphyg', 'reported', 'ユーザー指定URL'),
            'dataset_slug': FV('tipping', 'reported', 'ユーザー指定URL'),
            'url': FV('https://www.kaggle.com/datasets/jsphyg/tipping', 'reported', 'ユーザー指定URL'),
            'api_http_status': FV(api_status.get('http_status'), 'verified', 'Kaggle API実行セル'),
            'attempted_at_utc': FV(api_status['attempted_at_utc'], 'verified', '実行時刻（取得日ではない）'),
            'retrieved_file_name': FV(None, 'unknown', 'ファイル一覧取得不可'),
            'retrieved_at_utc': FV(None, 'unknown', '未取得'),
            'sha256': FV(None, 'unknown', '未取得のため算出不可'),
            'license': FV(None, 'unknown', '原データのライセンス未確認')},
        dataset_scope={
            'n': FV(None, 'unknown', '未取得'),
            'observation_unit': FV(None, 'unknown', 'データ辞書未確認'),
            'population': FV(None, 'unknown', 'データ辞書未確認'),
            'observation_period': FV(None, 'unknown', 'データ辞書未確認')},
        variables={name: {'definition': FV(None, 'unknown', '想定する列名だが実ファイル未確認'),
                          'unit': FV(None, 'unknown', '実ファイル・データ辞書未確認')}
                   for name in ['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']},
        transformations=({'planned_only': 'tip_rate_pct = 100 * tip / total_bill; total_bill > 0が必要'},))
    display(asdict(data_manifest))
    display({'unresolved_fields': [(key, asdict(value)) for key, value in data_manifest.unresolved_fields()],
             'inferred_fields': [], 'analysis_data_available': False})
    assumption_manifest = analysis_assumptions.AnalysisAssumptionManifest(
        analysis_scope={'intended': 'チップ額・チップ率の探索的関連分析', 'executed': 'API取得可否のみ'},
        assumptions=(
            analysis_assumptions.Assumption('target_available', '指定データをAPIで取得できる', 'rejected',
                impact_if_false='分析、品質検査、感度分析を実施できない', conclusion_critical=True),
            analysis_assumptions.Assumption('no_substitution', '別owner・外部CSVへ置換しない', 'verified',
                impact_if_false='ユーザー指定の出所条件を満たさない'),
            analysis_assumptions.Assumption('causal_identification', '因果効果を識別できる', 'rejected',
                impact_if_false='因果効果として解釈できない')),
        causal_scope='descriptive', sampling=None)
    assumption_findings = analysis_assumptions.check_manifest(assumption_manifest)
    display(asdict(assumption_manifest))
    display({'assumption_findings': [asdict(f) for f in assumption_findings]})
    display({'applicability': [
        {'feature': 'データ読込み・クリーニング・EDA・相関・回帰', 'status': 'not_applicable', 'reason': 'データ未取得'},
        {'feature': '意味的異常検査', 'status': 'not_applicable', 'reason': '検査対象DataFrameなし。正の会計額、非負のtip、正の整数人数、非欠損・カテゴリ制約は取得後に検査予定'},
        {'feature': '感度分析', 'status': 'not_applicable', 'reason': '主分析なし。取得後に全行／完全一致行除去／率の極端値除外、Pearson／Spearman、率モデル／額モデルを比較する予定。実行したと扱わない'},
        {'feature': '独立データ比較・異常の外部検証', 'status': 'not_applicable', 'reason': '第二の独立データ未提供。外部ソースへのフォールバックは禁止'},
        {'feature': '図表・visual_audit', 'status': 'audit_only', 'reason': '図表0件。監査はvisual_audit=Trueで実施するが可読性確認をしたことにはしない'},
        {'feature': 'mcp_gateway.run_and_record', 'status': 'not_used', 'reason': 'ホストのJupyter MCPツールで直接セル実行。カーネル内から呼ぶMCPClientは供給されていないため、模擬クライアントやローカルexecで代替しない'}]})


{'source': {'owner': {'value': 'jsphyg',
   'status': 'reported',
   'source': 'ユーザー指定URL'},
  'dataset_slug': {'value': 'tipping',
   'status': 'reported',
   'source': 'ユーザー指定URL'},
  'url': {'value': 'https://www.kaggle.com/datasets/jsphyg/tipping',
   'status': 'reported',
   'source': 'ユーザー指定URL'},
  'api_http_status': {'value': 403,
   'status': 'verified',
   'source': 'Kaggle API実行セル'},
  'attempted_at_utc': {'value': '2026-10-01T16:10:16.802184+00:00',
   'status': 'verified',
   'source': '実行時刻（取得日ではない）'},
  'retrieved_file_name': {'value': None,
   'status': 'unknown',
   'source': 'ファイル一覧取得不可'},
  'retrieved_at_utc': {'value': None, 'status': 'unknown', 'source': '未取得'},
  'sha256': {'value': None, 'status': 'unknown', 'source': '未取得のため算出不可'},
  'license': {'value': None, 'status': 'unknown', 'source': '原データのライセンス未確認'}},
 'dataset_scope': {'n': {'value': None, 'status': 'unknown', 'source': '未取得'},
  'observation_unit': {'value': None,
   'status': 'unknown',
   'source': '

{'unresolved_fields': [('source.retrieved_file_name',
   {'value': None, 'status': 'unknown', 'source': 'ファイル一覧取得不可'}),
  ('source.retrieved_at_utc',
   {'value': None, 'status': 'unknown', 'source': '未取得'}),
  ('source.sha256',
   {'value': None, 'status': 'unknown', 'source': '未取得のため算出不可'}),
  ('source.license',
   {'value': None, 'status': 'unknown', 'source': '原データのライセンス未確認'}),
  ('dataset_scope.n', {'value': None, 'status': 'unknown', 'source': '未取得'}),
  ('dataset_scope.observation_unit',
   {'value': None, 'status': 'unknown', 'source': 'データ辞書未確認'}),
  ('dataset_scope.population',
   {'value': None, 'status': 'unknown', 'source': 'データ辞書未確認'}),
  ('dataset_scope.observation_period',
   {'value': None, 'status': 'unknown', 'source': 'データ辞書未確認'}),
  ('variables.total_bill.definition',
   {'value': None, 'status': 'unknown', 'source': '想定する列名だが実ファイル未確認'}),
  ('variables.total_bill.unit',
   {'value': None, 'status': 'unknown', 'source': '実ファイル・データ辞書未確認'}),
  ('variables.tip.definiti

{'analysis_scope': {'intended': 'チップ額・チップ率の探索的関連分析', 'executed': 'API取得可否のみ'},
 'assumptions': ({'id': 'target_available',
   'statement': '指定データをAPIで取得できる',
   'status': 'rejected',
   'evidence_cell': None,
   'impact_if_false': '分析、品質検査、感度分析を実施できない',
   'conclusion_critical': True},
  {'id': 'no_substitution',
   'statement': '別owner・外部CSVへ置換しない',
   'status': 'verified',
   'evidence_cell': None,
   'impact_if_false': 'ユーザー指定の出所条件を満たさない',
   'conclusion_critical': False},
  {'id': 'causal_identification',
   'statement': '因果効果を識別できる',
   'status': 'rejected',
   'evidence_cell': None,
   'impact_if_false': '因果効果として解釈できない',
   'conclusion_critical': False}),
 'causal_scope': 'descriptive',
 'sampling': None}

{'assumption_findings': [{'code': 'unresolved_conclusion_critical_assumption',
   'severity': 'warning',
   'message': "Conclusion-critical assumption 'target_available' has status 'rejected', not verified/tested.",
   'assumption_id': 'target_available'}]}

{'applicability': [{'feature': 'データ読込み・クリーニング・EDA・相関・回帰',
   'status': 'not_applicable',
   'reason': 'データ未取得'},
  {'feature': '意味的異常検査',
   'status': 'not_applicable',
   'reason': '検査対象DataFrameなし。正の会計額、非負のtip、正の整数人数、非欠損・カテゴリ制約は取得後に検査予定'},
  {'feature': '感度分析',
   'status': 'not_applicable',
   'reason': '主分析なし。取得後に全行／完全一致行除去／率の極端値除外、Pearson／Spearman、率モデル／額モデルを比較する予定。実行したと扱わない'},
  {'feature': '独立データ比較・異常の外部検証',
   'status': 'not_applicable',
   'reason': '第二の独立データ未提供。外部ソースへのフォールバックは禁止'},
  {'feature': '図表・visual_audit',
   'status': 'audit_only',
   'reason': '図表0件。監査はvisual_audit=Trueで実施するが可読性確認をしたことにはしない'},
  {'feature': 'mcp_gateway.run_and_record',
   'status': 'not_used',
   'reason': 'ホストのJupyter MCPツールで直接セル実行。カーネル内から呼ぶMCPClientは供給されていないため、模擬クライアントやローカルexecで代替しない'}]}

## 使用したJupytermindモジュール

初回の直接利用履歴はmetadata `followup.original_sections` に保持した。以下は初回と今回の直接利用を区別した一覧。間接importを「使用済み」に数えない。

| 直接使用したモジュール | 関数・型 | 使用目的・場所 |
|---|---|---|
| `ai_data_scientist.project_manager` | `resolve_project`, `ensure_notebook`, `ensure_data_dir`, `enqueue_write`, `ProjectHandle` | 初回・今回のパス解決、保存先作成、制御Notebookからの直列保存。今回の一時的な監査再現用Notebookのハンドル作成も含む |
| `ai_data_scientist.language_router` | `detect_language` | 初回設定と今回制御セルで日本語を選択 |
| `ai_data_scientist.lifecycle` | `register_run`, `mark_execution_start`, `mark_execution_end`, `mark_write_start`, `mark_write_end`, `is_cancel_requested`, `mark_failed`, `mark_completed`, `get_run_status`, `wait_for_quiescence` | 初回分析のfailedと今回の再評価completedを別runで管理。制御保存フェーズを含む |
| `ai_data_scientist.data_definition` | `FieldValue`, `build_manifest`, `DataDefinitionManifest.unresolved_fields` | 初回manifest再実行、今回の定義不明項目の再確認 |
| `ai_data_scientist.analysis_assumptions` | `Assumption`, `AnalysisAssumptionManifest`, `check_manifest` | 初回manifest再実行、今回の結論に必須の取得前提不成立の再照合 |
| `ai_data_scientist.insight_engine` | `extract_cited_value`, `record_insight` | 初回・今回の制御セルで実出力からHTTPコードを抽出し、実行カウントを伴う停止結論を記録 |
| `ai_data_scientist.notebook_audit` | `audit_notebook(visual_audit=True)` | 全セル再実行後の証拠・図表監査。今回の見出し付き不一致証拠の単体再現も含む |

`ingestion`, `cleaning`, `eda`, `stats_analysis`, `visualization`, `anomaly_detection`, `data_quality`, `sensitivity`, `dataset_validation`, `mcp_gateway` は未使用。実データがないため統計・異常検査・感度分析の実行を装わない。実行経路はホストのJupyter MCPであり、`mcp_gateway.run_and_record` の直接呼出し実績とは区別する。Kaggle SDK、nbformat等はJupytermindモジュールに含めない。


## Jupytermind改善候補

再評価後の候補は **1件：見出し付きevidenceの監査漏れ**。外部ツール固有・原因未分離の問題は候補数から除外した。ソース修正は今回の分析再評価の対象外。

| 観点 | 再評価・期待する受け入れ条件 |
|---|---|
| 対象 | `ai_data_scientist.notebook_audit.audit_notebook` のMarkdown候補判定。`#` で始まるセルを除外するため、明示的evidenceも未検査になる |
| 再現性 | 今回、Kaggle・MCP同期を介さない一時Notebookで再現。コード出力403に対し引用999を持つ同一Markdownで、見出し先頭では `ok=True`, `insight_cell_count=0`、太字先頭だけに変えると `ok=False`, `insight_cell_count=1`。詳細をmetadata `followup.heading_reproduction` に保存 |
| 利用者への影響 | 見出し付き結論の古い・誤った引用が監査合格として渡される。分析の証拠追跡と再実行後の信頼性に影響。今回の取得失敗の原因ではない |
| 受け入れ条件 | 見出し・太字・通常文の各形式で、明示evidenceを同数検査する。一致引用は合格、不一致値・不存在実行カウント・不正JSON・必須キー欠落は全形式で不合格。証拠のない計画見出しは引き続き結論と誤認しない。既存の非見出し検査とvisual_auditを維持する |
| 今回の回避・確認 | 実結論は太字先頭のevidenceセルで記録。最終監査に加え、見出しに関係なく全evidence fenceを個別に実出力へ照合する |

### 候補から除外した事象

- Kaggle HTTP 403、公開状態・権限制約：外部API側。削除・非公開化などの原因は断定しない。
- 旧「実行中の自己書込みとMCP出力同期」：旧観測ログをmetadataに保持するが、Jupytermind単体での再現がなく外部MCP保存競合でも説明可能。Jupytermind候補から除外し、対象セル実行終了後に制御Notebookから保存する運用上の注意だけ残す。
- MCPの未作成親ディレクトリへの作成拒否、検索先頭ページだけによる未発見：外部ツール制約／実行手順の問題。Jupytermind固有の候補に含めない。


## 反復依頼履歴

反復回数：0／最大3回。初回分析に必要な原データを取得できていないため、分析結果の読み直しや自然言語の追加分析依頼は作成・実行していない。取得可否の再確認は分析の反復と数えない。

- 選定理由：HTTP 403が観測された段階で、ユーザーの停止条件を優先した。
- 結果・証拠：Kaggle APIセルの認証成功、一般検索20件、owner限定検索0件、ファイルAPIのHTTP 403。
- 残る限界：指定データの現在の公開状態とアクセス拒否の原因は不明。チップ額やチップ率に関する観察結果はない。

「価値ある追加分析がない」という結論ではなく、「原データ未取得のため追加分析が適用不能」という停止である。


### 今回の再評価履歴（v020-exp-04-followup）

元の0回は初回の履歴として保持する。今回の指定取得可否の再確認を保守的に追加依頼1回と数え、初回からの合計は1／3回。自然言語の依頼文・選定理由・実行対象・停止条件はフォローアップ節に記録した。初回の取得停止時点にAI生成の分析反復は存在しなかった。

追加依頼1の結果：指定ファイルAPIのHTTP 403を再現し、取得不可。追加の統計分析は適用不能として停止。再実行の修正検証は同じ依頼の再実行であり新規依頼には数えない。証拠：code index 3（execution_count=2）と13（execution_count=5）、結論evidence index 14。


## 再実行可能性・notebook_audit・ライフサイクル

最終確認はカーネルを再起動し、全コードセルを上からJupyter MCPで再実行する。停止状態を再現できることと、データ分析が成功したことを区別する。図表0件の監査は図表の品質検証ではない。自己監査セルは実行中なので、監査直後にホストから読み取り専用の最終監査も行いNotebook metadataに保存する。

再実行手順：カーネル再起動後、code index 1 → 3 → 6 → 11の順で実行する。証拠のexecution_countは再起動後の実行結果に照合する。監査セル実行中には同じNotebookを書き換えない。最終監査・状態保存はセル終了後に制御Notebookから行う。

### 初回保存時の実測結果（過去の記録）

新規カーネルで4／4コードセルを上から再実行。KaggleファイルAPIのHTTP 403を再現した。全セル終了後の `notebook_audit(visual_audit=True)` は `ok=True`、未実行0、セルエラー0、停止記録の証拠1件、図表0件、visual_findings 0件。ライフサイクルは `failed`、実行中・書込み待ち・保持ロックはいずれも0。データ分析は未実施で、監査合格は分析成功を意味しない。取得前提不成立のassumption警告1件を残した。最終の詳細はNotebook metadata `notebook_audit`、`lifecycle_status`、`phase_log`、`reexecution`、診断ログ2種に記録。

### 今回の再評価の確認方法

カーネル再起動後、全code index 1 → 3 → 6 → 11 → 13を順に実行する。追加依頼はindex 3と13で実行し、初回runのfailedとfollowup runのcompletedを区別する。最後に別制御Notebookで保存済みの全コード出力と全evidence fenceを確認し、visual_audit=Trueの監査結果とlifecycleをmetadataへ保存する。対象Notebook自身の実行中書込みは禁止。取得ファイル・SHA-256の欠如を明示し、監査合格を分析成功に読み替えない。


### 今回の最終保存時の実測結果（v020-exp-04-followup）

新規カーネルで全5／5コードセルを上から再実行（index 1, 3, 6, 11, 13、execution_count 1–5）。HTTP 403を再現、Kaggle指定owner/slugはjsphyg/tipping、取得ファイル0、取得日時・SHA-256はnull。全セル終了・根拠付き結論追加後の `notebook_audit(visual_audit=True)` はok=True、未実行0、エラー0、証拠2件、図表0、findings 0、visual_findings 0。見出しを含む全evidence fenceの実出力照合も2／2成功。followup lifecycleはcompleted、初回分析はfailed、両runの実行中・書込み待ち・保持ロックは0。取得前提不成立のassumption警告1件は分析上の制約として保持。図表の可読性・感度の安定性は検査できていない。詳細はmetadata `followup` と最終監査記録に保存。


In [4]:
from ai_data_scientist import notebook_audit
with phase('Notebook・図表監査'):
    audit = notebook_audit.audit_notebook(handle.notebook_path, visual_audit=True)
    display({'notebook_audit': asdict(audit), 'ok': audit.ok, 'visual_audit': True,
             'chart_count': len(audit.chart_cell_indices), 'analysis_completed': False})
    if not audit.ok:
        raise RuntimeError('Notebook監査で未解決の問題が検出されました')
    if api_status['status'] != 'failed' or api_status.get('http_status') != 403:
        raise RuntimeError('データ取得条件が変わったため停止記録の更新または分析の再計画が必要です')
lifecycle.mark_failed(RUN_ID)
status = lifecycle.wait_for_quiescence(RUN_ID, timeout_s=5)
print('停止ログの再実行成功とデータ分析の成功は別。実験はHTTP 403によりfailed。')
display({'lifecycle': asdict(status), 'phase_log': phase_log,
         'assumption_findings': [asdict(f) for f in assumption_findings]})


停止ログの再実行成功とデータ分析の成功は別。実験はHTTP 403によりfailed。


{'notebook_audit': {'path': '/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-04-tips/notebooks/kaggle-v020-kaggle-exp-04-tips.ipynb',
  'nbformat_valid': True,
  'code_cell_count': 5,
  'executed_code_cell_count': 4,
  'unexecuted_cell_indices': (),
  'error_cell_indices': (),
  'chart_cell_indices': (),
  'insight_cell_count': 1,
  'findings': ({'severity': 'warning',
    'message': 'Trailing cell invokes audit_notebook and has not finished executing yet; excluded from unexecuted-cell findings.',
    'cell_index': 13},),
  'visual_findings': ()},
 'ok': True,
 'visual_audit': True,
 'chart_count': 0,
 'analysis_completed': False}

{'lifecycle': {'run_id': 'v020-exp-04',
  'state': 'failed',
  'active_cell_executions': 0,
  'pending_notebook_writes': 0,
  'locks_held': 0,
  'notebook_path': '/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-04-tips/notebooks/kaggle-v020-kaggle-exp-04-tips.ipynb',
  'reason': None},
 'phase_log': [{'phase': 'Kaggle公開データ検索・取得',
   'event': 'start',
   'utc': '2026-10-01T16:10:15.911040+00:00'},
  {'phase': 'Kaggle公開データ検索・取得',
   'event': 'end',
   'utc': '2026-10-01T16:10:17.940263+00:00'},
  {'phase': '未取得データの定義・分析前提manifest',
   'event': 'start',
   'utc': '2026-10-01T16:10:26.767200+00:00'},
  {'phase': '未取得データの定義・分析前提manifest',
   'event': 'end',
   'utc': '2026-10-01T16:10:26.774690+00:00'},
  {'phase': 'Notebook・図表監査',
   'event': 'start',
   'utc': '2026-10-01T16:10:30.579335+00:00'},
  {'phase': 'Notebook・図表監査',
   'event': 'end',
   'utc': '2026-10-01T16:10:30.653862+00:00'}],
 'assumption_findings': [{'code': 'unresolved_conclusion_critical_assum

## フォローアップ再評価：v020-exp-04-followup

既存12セルのID・順序・コードは保持した。初回run `v020-exp-04` の停止と今回の再評価を区別する。依頼された問いは「チップ額・チップ率に関係する特徴」。初回はデータ取得前に停止しており、この問いに未回答。AI生成反復依頼は元記録では0件で、既存反復結果を評価できる対象もない。計画や監査合格を分析結果と見なさない。

| 再評価観点 | 判断・残る制約 |
|---|---|
| データ定義 | owner/slugは指定値であって取得確認ではない。7列は想定名、通貨、会計額にtipが含まれるか、人数の意味、観測単位・抽出母集団・期間は未確認。n=244やUSD等の既知のTips情報を持ち込まない。未取得のファイル名・取得時刻・SHA-256はnull |
| 分析前提 | 取得可能性は結論に必須だがrejected。別データに置換しない条件は保持。仮に取得できても非無作為標本、店・担当者・顧客等の未観測交絡、カテゴリ不均衡、観測の独立性未確認を扱う必要がある。因果効果は識別していない |
| 異常値 | 実データなしで検査不能。取得後の正の会計額、非負tip、正の整数人数、カテゴリ・欠損・重複、極端率の確認は計画のみ。極端率や完全一致行を自動誤記・重複観測と認定しない。独立参照データも未提供 |
| 感度 | 主分析なし、stableや最大偏差の数値は生成不能。取得後の全行／完全一致行除去／極端率除外、Pearson／Spearman、HC3付き額モデル／率モデルを比較する計画は未実行。100×tip/total_billの分母共有、平均率と金額加重率の差も未検証 |
| 可視化 | 図表0件。散布図・分布・群別nと不確実性・回帰診断は未作成。visual_findings=0は日本語可読性、軸、外れ値表現の検証成功を意味しない |
| 証拠manifest | HTTP 403はアクセス拒否の証拠だけで、データ不存在やtipの関係を証明しない。再起動後の実行カウント・引用値を実出力へ再照合。見出し付き証拠の監査漏れを補う全fence照合も行う |

### AI自身が作成した追加依頼1／最大3回

結論を変え得る唯一の操作可能な未解決点は、指定データの取得条件が回復したかどうか。取得再確認も今回は保守的に1回として数える。初回0＋今回1＝合計1。次の文を実行前に記録し、最終上から再実行のKaggleセル（index 3）と再評価セルで実行する。

> 指定Kaggle jsphyg/tippingについて、初回の取得失敗が現在も分析の結論を阻むか、同じ認証・owner限定検索・指定slugのファイルAPIを再実行して確認してください。別ownerや外部CSVへ置換しないでください。取得不能ならファイル名・取得日時・SHA-256を未取得として記録し、列定義、分析前提、異常値、感度、可視化、証拠manifestの適用不能と未解決点を区別してください。停止理由以外の統計的結論を生成せず、証拠参照とライフサイクルを検証してください。

成功時は停止結論を破棄して実ファイルに基づく分析を再計画する。失敗が続けば統計分析を追加しても根拠が得られないため2・3回目を作成しない。最終結果は続く実行出力・evidence・metadataに保存する。


In [5]:
from ai_data_scientist import insight_engine
import re
FOLLOWUP_RUN_ID = 'v020-exp-04-followup'
lifecycle.register_run(FOLLOWUP_RUN_ID, handle.notebook_path)
if lifecycle.is_cancel_requested(FOLLOWUP_RUN_ID):
    raise RuntimeError('再評価の取消し要求あり')
lifecycle.mark_execution_start(FOLLOWUP_RUN_ID)
try:
    if api_status['status'] != 'failed' or api_status.get('http_status') != 403:
        raise RuntimeError('取得条件が変化しました。停止結論を更新して実データ分析を再計画してください')
    if provenance is not None:
        raise RuntimeError('未取得の判断とprovenanceが矛盾しています')
    staged_files = sorted(p.name for p in data_dir.iterdir() if p.is_file())
    if staged_files:
        raise RuntimeError('保存済みファイルがあります。出所とSHA-256の検証なしに未取得と断定できません')
    unresolved = [(key, asdict(value)) for key, value in data_manifest.unresolved_fields()]
    findings = analysis_assumptions.check_manifest(assumption_manifest)
    if not any(f.assumption_id == 'target_available' for f in findings):
        raise RuntimeError('取得前提不成立の警告が見つかりません')
    saved = nbformat.read(handle.notebook_path, as_version=4)
    fence_checks = []
    for index, cell in enumerate(saved.cells):
        if cell.cell_type != 'markdown':
            continue
        for payload in re.findall(r'```evidence\n(.*?)\n```', cell.source, re.DOTALL):
            manifest = json.loads(payload)
            if not {'execution_count', 'cited_value', 'claim_type'} <= manifest.keys():
                raise RuntimeError('証拠manifestの必須キーが不足しています')
            matches = [c for c in saved.cells if c.cell_type == 'code' and c.execution_count == manifest['execution_count']]
            if len(matches) != 1:
                raise RuntimeError('証拠の実行カウントが一意に解決しません')
            output_text = '\n'.join(str(v) for o in matches[0].outputs for v in o.get('data', {}).values())
            if str(manifest['cited_value']) not in output_text:
                raise RuntimeError('証拠引用が実出力に存在しません')
            fence_checks.append({'markdown_index': index, **manifest, 'resolved': True})
    followup_report = {
        'run_id': FOLLOWUP_RUN_ID, 'initial_analysis_answered_question': False,
        'existing_ai_iteration_count': 0, 'new_request_count': 1, 'total_request_count': 1, 'max_requests': 3,
        'request_execution': '記録済み追加依頼1をKaggleセルindex 3と本セルで実行',
        'request_outcome': '取得不可を再現。問いは未回答、追加の統計分析は適用不能',
        'requested_owner': 'jsphyg', 'requested_slug': 'tipping', 'requested_ref': api_status['requested_ref'],
        'owner_slug_status': '指定値。データ実体の取得確認は未完了',
        'attempted_at_utc': api_status['attempted_at_utc'], 'http_status': api_status['http_status'],
        'retrieved_files': [], 'retrieved_at_utc': None, 'sha256': None,
        'staged_files': staged_files, 'analysis_completed': False,
        'unresolved_definition_fields': unresolved, 'assumption_findings': [asdict(f) for f in findings],
        'anomaly_detection': 'not_applicable_no_data', 'sensitivity': 'not_applicable_no_primary_analysis',
        'visual_quality': 'not_applicable_no_charts', 'evidence_checks': fence_checks,
        'stop_rule': '分析対象なし。2・3回目は作成しない。外部データへ置換しない'}
    display(followup_report)
    followup_audit = notebook_audit.audit_notebook(handle.notebook_path, visual_audit=True)
    display({'notebook_audit': asdict(followup_audit), 'ok': followup_audit.ok, 'visual_audit': True})
    if not followup_audit.ok:
        raise RuntimeError('再評価Notebookの監査に未解決の指摘があります')
except Exception:
    lifecycle.mark_failed(FOLLOWUP_RUN_ID)
    raise
finally:
    lifecycle.mark_execution_end(FOLLOWUP_RUN_ID)
lifecycle.mark_completed(FOLLOWUP_RUN_ID)
followup_status = lifecycle.wait_for_quiescence(FOLLOWUP_RUN_ID, timeout_s=5)
display({'followup_lifecycle': asdict(followup_status), 'initial_analysis_lifecycle': asdict(lifecycle.get_run_status(RUN_ID)), 'evaluation_completed': True, 'analysis_completed': False})


{'run_id': 'v020-exp-04-followup',
 'initial_analysis_answered_question': False,
 'existing_ai_iteration_count': 0,
 'new_request_count': 1,
 'total_request_count': 1,
 'max_requests': 3,
 'request_execution': '記録済み追加依頼1をKaggleセルindex 3と本セルで実行',
 'request_outcome': '取得不可を再現。問いは未回答、追加の統計分析は適用不能',
 'requested_owner': 'jsphyg',
 'requested_slug': 'tipping',
 'requested_ref': 'jsphyg/tipping',
 'owner_slug_status': '指定値。データ実体の取得確認は未完了',
 'attempted_at_utc': '2026-10-01T16:10:16.802184+00:00',
 'http_status': 403,
 'retrieved_files': [],
 'retrieved_at_utc': None,
 'sha256': None,
 'staged_files': [],
 'analysis_completed': False,
 'unresolved_definition_fields': [('source.retrieved_file_name',
   {'value': None, 'status': 'unknown', 'source': 'ファイル一覧取得不可'}),
  ('source.retrieved_at_utc',
   {'value': None, 'status': 'unknown', 'source': '未取得'}),
  ('source.sha256',
   {'value': None, 'status': 'unknown', 'source': '未取得のため算出不可'}),
  ('source.license',
   {'value': None, 'status': 'unknown',

{'notebook_audit': {'path': '/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-04-tips/notebooks/kaggle-v020-kaggle-exp-04-tips.ipynb',
  'nbformat_valid': True,
  'code_cell_count': 5,
  'executed_code_cell_count': 4,
  'unexecuted_cell_indices': (),
  'error_cell_indices': (),
  'chart_cell_indices': (),
  'insight_cell_count': 1,
  'findings': ({'severity': 'warning',
    'message': 'Trailing cell invokes audit_notebook and has not finished executing yet; excluded from unexecuted-cell findings.',
    'cell_index': 13},),
  'visual_findings': ()},
 'ok': True,
 'visual_audit': True}

{'followup_lifecycle': {'run_id': 'v020-exp-04-followup',
  'state': 'completed',
  'active_cell_executions': 0,
  'pending_notebook_writes': 0,
  'locks_held': 0,
  'notebook_path': '/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-04-tips/notebooks/kaggle-v020-kaggle-exp-04-tips.ipynb',
  'reason': None},
 'initial_analysis_lifecycle': {'run_id': 'v020-exp-04',
  'state': 'failed',
  'active_cell_executions': 0,
  'pending_notebook_writes': 0,
  'locks_held': 0,
  'notebook_path': '/home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-04-tips/notebooks/kaggle-v020-kaggle-exp-04-tips.ipynb',
  'reason': None},
 'evaluation_completed': True,
 'analysis_completed': False}

**フォローアップの結論（v020-exp-04-followup）**

指定 `jsphyg/tipping` のファイルAPIは再実行でもHTTP 403を返した。取得ファイルはなく、取得日時・SHA-256は未取得。初回と追加依頼1回はチップ額・チップ率の問いに答える実データ分析に至っていない。列定義、異常値、感度、図表を検証済みとは扱わず、別owner・外部CSVで補完しない。原データがないため2・3回目の分析依頼は実行しない。再評価タスクのcompletedと初回データ分析のfailedは別の状態である。

```evidence
{"execution_count":2,"cited_value":"403","claim_type":"descriptive"}
```